# 06 Generate Southern Ocean Subset DII Intermediates

Purpose: generate the Southern Ocean SPSS regional-exclusion point caches, feature weights, and DII curve provenance files.

Inputs:
- pCO2 LEAP reconstruction zarr
- SOCAT mask zarr
- RECCAP2 region mask NetCDF

Outputs:
- `SM_minus_SO_sample_points.csv`
- `SO_SPSS_points.csv`
- `Finalweights_SOCAT_minus_SO.txt`
- `Finalimbs_SOCAT_minus_SO.txt`
- `StandardII_imbs_SOCAT_minus_SO.txt`
- `StandardII_imbs_SO_SPSS.txt`

Estimated runtime: slow. This notebook learns feature weights on SOCAT observations excluding Southern Ocean SPSS, then evaluates DII on both the excluded-SOCAT sample and the Southern Ocean SPSS region.

Notes:
- This notebook is provenance material for the Southern Ocean regional-exclusion figures.
- This optional regeneration step requires the raw data and a compatible Python environment.
- Point caches are generated here so the map provenance is in the same notebook as the DII provenance.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from dadapy.feature_weighting import FeatureWeighting
from dadapy.metric_comparisons import MetricComparisons
from sklearn.preprocessing import StandardScaler

RECONSTRUCTION_ZARR = Path("../raw/pCO2_LEAP_fco2-residual-full-dataset-preML_198201-202412.zarr")
SOCAT_MASK_ZARR = Path("../raw/socat_mask_feb1982-dec2022.zarr")
REGION_MASK_FILE = Path("../raw/RECCAP2_region_masks_all_v20221025.nc")
OUTPUT_DIR = Path("../intermediates")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TIME_SLICE = slice("2020-01-01", "2022-12-31")
N_SAMPLE = 16000
RANDOM_STATE = 10
N_EPOCHS = 80


## Feature Definitions


In [ ]:
FEATURES = [
    "sst",
    "sst_anomaly",
    "sss",
    "sss_anomaly",
    "chl_log",
    "chl_log_anomaly",
    "mld_log",
    "xco2_trend",
    "A",
    "B",
    "C",
    "T0",
    "T1",
]
TARGET = "delta_fco2_1D"


## Region Configuration


In [ ]:
REGION_MASK_VARIABLE = "southern"
REGION_VALUE = 2
REGION_LABEL = "Southern Ocean SPSS"


## Helper Functions


In [ ]:
def add_lat_lon(frame):
    out = frame.copy()
    x = out["A"].to_numpy()
    y = out["B"].to_numpy()
    z = out["C"].to_numpy()
    radius = np.sqrt(x * x + y * y + z * z)
    x, y, z = x / radius, y / radius, z / radius
    out["lat_deg"] = np.degrees(np.arcsin(x))
    out["lon_deg"] = (np.degrees(-np.arctan2(y, z))) % 360 - 180
    return out

def shifted_region_mask(mask_file, variable, region_value):
    mask = xr.open_dataset(mask_file)[variable]
    mask = mask.roll(lon=180, roll_coords="lon")
    mask["lon"] = np.arange(-179.5, 180.5, 1)
    mask = mask.rename({"lon": "xlon", "lat": "ylat"})
    return mask == region_value

def load_base_data(reconstruction_zarr, socat_mask_zarr):
    ds = xr.open_dataset(reconstruction_zarr, engine="zarr")
    ds = ds.sel(time=slice("1982-02-01", "2022-12-31"))
    ds[TARGET] = ds["fco2"] - ds["xco2_trend"]
    ds = ds[FEATURES + [TARGET]]

    socat_mask = xr.open_dataset(socat_mask_zarr, engine="zarr")
    aligned_socat_mask = socat_mask.reindex(time=ds.time, method="nearest")
    ds_socat = ds.where(aligned_socat_mask.socat_mask == 1)
    return ds, ds_socat

def scale_frame(frame):
    scaler = StandardScaler()
    scaled = scaler.fit_transform(frame.loc[:, FEATURES + [TARGET]])
    return pd.DataFrame(scaled, columns=FEATURES + [TARGET])

def compute_standard_dii_curve(scaled_frame, final_weights, label):
    imbalances = []
    y = scaled_frame[[TARGET]].to_numpy()

    for k, weights in enumerate(final_weights):
        if np.any(np.isnan(weights)):
            print(label, "NaNs encountered for index", k)
            imbalances.append(np.nan)
            continue

        x = np.hstack([scaled_frame[FEATURES].to_numpy() * weights, y])
        comparison = MetricComparisons(x, maxk=x.shape[0] - 1)
        imbalance = comparison.return_inf_imb_two_selected_coords(
            coords1=range(x.shape[1] - 1),
            coords2=[-1],
        )[0]
        print(label, imbalance)
        imbalances.append(imbalance)

    return imbalances


## Load Raw Data and Select Southern Ocean SPSS


In [ ]:
ds, ds_socat = load_base_data(RECONSTRUCTION_ZARR, SOCAT_MASK_ZARR)
region_mask = shifted_region_mask(REGION_MASK_FILE, REGION_MASK_VARIABLE, REGION_VALUE)

region_frame = ds.where(region_mask).sel(time=TIME_SLICE).to_dataframe().dropna()
socat_minus_region_frame = ds_socat.sel(time=TIME_SLICE).where(~region_mask).to_dataframe().dropna()

region_frame.shape, socat_minus_region_frame.shape


## Sample, Scale, and Save Point Caches


In [ ]:
socat_minus_region_sample = socat_minus_region_frame.sample(n=N_SAMPLE, random_state=RANDOM_STATE)
socat_minus_region_scaled = scale_frame(socat_minus_region_sample)
region_scaled = scale_frame(region_frame)

add_lat_lon(socat_minus_region_sample[["A", "B", "C"]]).to_csv(
    OUTPUT_DIR / "SM_minus_SO_sample_points.csv",
    index=False,
)
add_lat_lon(region_frame[["A", "B", "C"]]).to_csv(
    OUTPUT_DIR / "SO_SPSS_points.csv",
    index=False,
)

socat_minus_region_scaled.shape, region_scaled.shape


## Learn Feature Weights on SOCAT Excluding the Region


In [ ]:
target = FeatureWeighting(socat_minus_region_scaled[[TARGET]].to_numpy(), verbose=True)
inputs = FeatureWeighting(socat_minus_region_scaled[FEATURES].to_numpy(), verbose=True)

final_imbs, final_weights = inputs.return_backward_greedy_dii_elimination(
    target_data=target,
    initial_weights=None,
    n_epochs=N_EPOCHS,
    learning_rate=None,
    decaying_lr="cos",
)

np.savetxt(OUTPUT_DIR / "Finalweights_SOCAT_minus_SO.txt", final_weights)
np.savetxt(OUTPUT_DIR / "Finalimbs_SOCAT_minus_SO.txt", final_imbs)


## Compute DII Curve on SOCAT Excluding the Region


In [ ]:
imbs_minus_region = compute_standard_dii_curve(
    socat_minus_region_scaled,
    final_weights,
    label="SOCAT_minus_SO",
)
np.savetxt(OUTPUT_DIR / "StandardII_imbs_SOCAT_minus_SO.txt", imbs_minus_region)


## Compute DII Curve on Southern Ocean SPSS


In [ ]:
imbs_region = compute_standard_dii_curve(
    region_scaled,
    final_weights,
    label="SO_SPSS",
)
np.savetxt(OUTPUT_DIR / "StandardII_imbs_SO_SPSS.txt", imbs_region)


## Outputs Created

Quick check:


In [ ]:
expected_outputs = ['SM_minus_SO_sample_points.csv', 'SO_SPSS_points.csv', 'Finalweights_SOCAT_minus_SO.txt', 'Finalimbs_SOCAT_minus_SO.txt', 'StandardII_imbs_SOCAT_minus_SO.txt', 'StandardII_imbs_SO_SPSS.txt']

for filename in expected_outputs:
    path = OUTPUT_DIR / filename
    print(filename, "OK" if path.exists() else "MISSING")
